In [3]:
import os
import re
import glob
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# USER INPUT
# ============================================================
INPUT_ROOT = r"/Users/parth/Library/CloudStorage/Box-Box/WHT Datasets/03_gravity_removed"
OUTPUT_DIR = r"/Users/parth/Downloads/dataset_comparison_recent"

# ============================================================
# SPEED / MEMORY SETTINGS
# ============================================================
RANDOM_SEED = 42

MAX_FILES_PER_DATASET = 12
MAX_POINTS_PER_FILE = 2500
MAX_POINTS_PER_DATASET_PLOT = 12000
MAX_TOTAL_POINTS_ALL_PLOT = 50000

FILE_EXTENSIONS = ("*.csv", "*.CSV")

USE_PATH_BASED_WALKING_FILTER = True
USE_ACTIVITY_COLUMN_FALLBACK = False

TARGET_SINGLE_DATASET = "YARETA"

# ============================================================
# COLUMN PRIORITY
# ============================================================
SIGNAL_PRIORITY = [
    ("shank", "acc", "x"),
    ("shank", "acc", "y"),
    ("shank", "acc", "z"),
    ("shank", "gyr", "x"),
    ("shank", "gyr", "y"),
    ("shank", "gyr", "z"),
    ("thigh", "acc", "x"),
    ("thigh", "acc", "y"),
    ("thigh", "acc", "z"),
    ("thigh", "gyr", "x"),
    ("thigh", "gyr", "y"),
    ("thigh", "gyr", "z"),
]

# ============================================================
# SENSOR KEYWORDS
# ============================================================
THIGH_KEYWORDS = ["thigh", "upperleg", "upper_leg", "femur"]
SHANK_KEYWORDS = ["shank", "lowerleg", "lower_leg", "tibia"]

ACC_KEYWORDS = ["acc", "accelerometer", "accel"]
GYR_KEYWORDS = ["gyr", "gyro", "gyroscope"]
AXIS_KEYWORDS = ["x", "y", "z"]

# ============================================================
# ACTIVITY KEYWORDS
# ============================================================
ACTIVITY_COLUMN_CANDIDATES = [
    "activity", "activities", "label", "labels", "task", "tasks",
    "locomotion", "movement", "motion", "class", "state", "trial", "action"
]

WALKING_LABEL_KEYWORDS = [
    "walk",
    "walking",
    "levelwalking",
    "level walking",
    "level_ground_walking",
    "level ground walking",
    "treadmill",
    "treadmill walking",
    "overground walking",
    "gait",
    "levelground"
]

PATH_WALKING_KEYWORDS = [
    "walk",
    "walking",
    "levelground",
    "level_ground",
    "treadmill",
    "gait"
]

PATH_EXCLUDE_KEYWORDS = [
    "stair",
    "stairs",
    "ramp",
    "jump",
    "sit",
    "stand",
    "turn",
    "transition"
]

# ============================================================
# HELPERS
# ============================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def resolve_output_dir(path):
    """
    Try to create and use the requested output directory.
    If that fails, fall back to a folder in the current working directory.
    """
    try:
        os.makedirs(path, exist_ok=True)

        test_file = os.path.join(path, "__write_test__.tmp")
        with open(test_file, "w") as f:
            f.write("test")
        os.remove(test_file)

        return path
    except Exception as e:
        fallback = os.path.join(os.getcwd(), "dataset_comparison_recent")
        os.makedirs(fallback, exist_ok=True)
        print(f"Requested OUTPUT_DIR could not be used: {path}")
        print(f"Reason: {e}")
        print(f"Using fallback OUTPUT_DIR instead: {fallback}")
        return fallback

def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")

def normalize_text(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()

def get_all_csv_files(input_root):
    if os.path.isfile(input_root) and input_root.lower().endswith(".csv"):
        return [input_root]

    csv_files = []
    for ext in FILE_EXTENSIONS:
        csv_files.extend(glob.glob(os.path.join(input_root, "**", ext), recursive=True))
    return sorted(csv_files)

def infer_dataset_name(file_path, root_path):
    rel = os.path.relpath(file_path, root_path)
    parts = rel.split(os.sep)
    if len(parts) >= 2:
        return parts[0]
    return os.path.basename(root_path)

def is_numeric_dtype_from_sample(series):
    try:
        pd.to_numeric(series.dropna().head(20), errors="raise")
        return True
    except Exception:
        return False

def stable_sample_indices(n, max_n, seed=42):
    if n <= max_n:
        return np.arange(n)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(n, size=max_n, replace=False))
    return idx

def robust_limits(values, low=1, high=99, pad_frac=0.08):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (-1, 1)
    lo = np.percentile(values, low)
    hi = np.percentile(values, high)
    if np.isclose(lo, hi):
        lo -= 1
        hi += 1
    pad = (hi - lo) * pad_frac
    return lo - pad, hi + pad

# ============================================================
# WALKING DETECTION
# ============================================================
def path_looks_like_walking(file_path):
    p = normalize_text(file_path)
    has_walk = any(k.replace("_", " ") in p for k in PATH_WALKING_KEYWORDS)
    has_excluded = any(k in p for k in PATH_EXCLUDE_KEYWORDS)
    return has_walk and not has_excluded

def find_activity_column(columns):
    norm_cols = {c: normalize_name(c) for c in columns}
    candidates = [normalize_name(x) for x in ACTIVITY_COLUMN_CANDIDATES]

    for c, nc in norm_cols.items():
        if nc in candidates:
            return c

    for c, nc in norm_cols.items():
        if any(cc in nc for cc in candidates):
            return c

    return None

def is_walking_label(value):
    s = normalize_text(value)
    if s == "" or s == "nan":
        return False
    return any(normalize_text(k) in s for k in WALKING_LABEL_KEYWORDS)

# ============================================================
# COLUMN CLASSIFICATION
# ============================================================
def classify_sensor_column(col_name):
    c = normalize_name(col_name)

    sensor_type = None
    signal_type = None
    axis = None

    if any(k in c for k in THIGH_KEYWORDS):
        sensor_type = "thigh"
    elif any(k in c for k in SHANK_KEYWORDS):
        sensor_type = "shank"

    if any(k in c for k in ACC_KEYWORDS):
        signal_type = "acc"
    elif any(k in c for k in GYR_KEYWORDS):
        signal_type = "gyr"

    for a in AXIS_KEYWORDS:
        if re.search(rf"(?:^|_){a}(?:$|_)", c) or c.endswith(a):
            axis = a
            break

    return sensor_type, signal_type, axis

def scan_file_header_for_best_column(csv_path):
    try:
        sample = pd.read_csv(csv_path, nrows=20)
    except Exception:
        return None, None, None

    columns = list(sample.columns)
    activity_col = find_activity_column(columns)

    classified = []
    for col in columns:
        sensor_type, signal_type, axis = classify_sensor_column(col)
        if sensor_type is None or signal_type is None or axis is None:
            continue
        if not is_numeric_dtype_from_sample(sample[col]):
            continue
        classified.append((col, sensor_type, signal_type, axis))

    if not classified:
        return None, activity_col, columns

    for wanted_sensor, wanted_signal, wanted_axis in SIGNAL_PRIORITY:
        for col, sensor_type, signal_type, axis in classified:
            if (sensor_type, signal_type, axis) == (wanted_sensor, wanted_signal, wanted_axis):
                return col, activity_col, columns

    col, _, _, _ = classified[0]
    return col, activity_col, columns

# ============================================================
# FAST FILE READING
# ============================================================
def load_single_signal_fast(csv_path, signal_col, activity_col=None):
    usecols = [signal_col]
    if activity_col is not None and activity_col != signal_col:
        usecols.append(activity_col)

    try:
        df = pd.read_csv(csv_path, usecols=usecols)
    except Exception:
        return None

    if signal_col not in df.columns:
        return None

    df[signal_col] = pd.to_numeric(df[signal_col], errors="coerce")
    df = df.dropna(subset=[signal_col])

    if len(df) == 0:
        return None

    if activity_col is not None and activity_col in df.columns and USE_ACTIVITY_COLUMN_FALLBACK:
        mask = df[activity_col].astype(str).apply(is_walking_label)
        df = df.loc[mask].copy()
        if len(df) == 0:
            return None

    return df[[signal_col]].reset_index(drop=True)

def sample_signal_points(df_signal, signal_col, max_points, seed):
    values = df_signal[signal_col].to_numpy()
    n = len(values)
    idx = stable_sample_indices(n, min(max_points, n), seed=seed)
    return values[idx]

# ============================================================
# DATA COLLECTION
# ============================================================
def collect_fast_dataset_points(input_root):
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    csv_files = get_all_csv_files(input_root)
    if len(csv_files) == 0:
        print("No CSV files found.")
        return {}, []

    root_path = input_root if os.path.isdir(input_root) else os.path.dirname(input_root)

    files_by_dataset = defaultdict(list)
    for fp in csv_files:
        ds = infer_dataset_name(fp, root_path)
        files_by_dataset[ds].append(fp)

    dataset_points = {}
    summary_rows = []

    for dataset_name, dataset_files in sorted(files_by_dataset.items()):
        print(f"\nProcessing dataset: {dataset_name} | total files found: {len(dataset_files)}")

        if USE_PATH_BASED_WALKING_FILTER:
            walking_candidates = [fp for fp in dataset_files if path_looks_like_walking(fp)]
        else:
            walking_candidates = list(dataset_files)

        if len(walking_candidates) == 0:
            print("  No walking-like files found from file paths.")
            summary_rows.append({
                "dataset": dataset_name,
                "files_found": len(dataset_files),
                "walking_candidates": 0,
                "used_files": 0,
                "chosen_signal": "",
                "status": "no_walking_candidates"
            })
            continue

        random.Random(RANDOM_SEED).shuffle(walking_candidates)
        selected_files = walking_candidates[:MAX_FILES_PER_DATASET]

        all_x = []
        all_y = []
        chosen_signal_name = None
        used_files = 0

        for local_i, csv_path in enumerate(selected_files):
            signal_col, activity_col, _ = scan_file_header_for_best_column(csv_path)

            if signal_col is None:
                continue

            if chosen_signal_name is None:
                chosen_signal_name = signal_col

            df_signal = load_single_signal_fast(
                csv_path=csv_path,
                signal_col=signal_col,
                activity_col=activity_col
            )

            if df_signal is None or len(df_signal) == 0:
                continue

            vals = sample_signal_points(
                df_signal=df_signal,
                signal_col=signal_col,
                max_points=MAX_POINTS_PER_FILE,
                seed=RANDOM_SEED + local_i
            )

            x = np.arange(len(vals)) + len(np.concatenate(all_x)) if len(all_x) > 0 else np.arange(len(vals))
            all_x.append(x)
            all_y.append(vals)
            used_files += 1

        if used_files == 0:
            print("  No usable files after filtering.")
            summary_rows.append({
                "dataset": dataset_name,
                "files_found": len(dataset_files),
                "walking_candidates": len(walking_candidates),
                "used_files": 0,
                "chosen_signal": "",
                "status": "no_usable_files"
            })
            continue

        x = np.concatenate(all_x).astype(np.int32)
        y = np.concatenate(all_y).astype(np.float32)

        if len(y) > MAX_POINTS_PER_DATASET_PLOT:
            keep_idx = stable_sample_indices(len(y), MAX_POINTS_PER_DATASET_PLOT, seed=RANDOM_SEED)
            x = x[keep_idx]
            y = y[keep_idx]

            order = np.argsort(x)
            x = x[order]
            y = y[order]

        dataset_points[dataset_name] = {
            "x": x,
            "y": y,
            "n_points": len(y),
            "used_files": used_files,
            "signal_name": chosen_signal_name if chosen_signal_name is not None else "unknown_signal"
        }

        print(f"  Walking candidates: {len(walking_candidates)}")
        print(f"  Used files: {used_files}")
        print(f"  Points kept: {len(y)}")
        print(f"  Signal used: {dataset_points[dataset_name]['signal_name']}")

        summary_rows.append({
            "dataset": dataset_name,
            "files_found": len(dataset_files),
            "walking_candidates": len(walking_candidates),
            "used_files": used_files,
            "chosen_signal": dataset_points[dataset_name]["signal_name"],
            "points_kept": len(y),
            "status": "processed"
        })

    return dataset_points, summary_rows

# ============================================================
# PLOTTING
# ============================================================
def get_dataset_colors(dataset_names):
    cmap = plt.get_cmap("tab10")
    colors = {}
    for i, ds in enumerate(dataset_names):
        colors[ds] = cmap(i % 10)
    return colors

def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.22, linewidth=0.8)
    ax.tick_params(labelsize=10)

def plot_single_dataset(dataset_points, dataset_name, color, save_path):
    if dataset_name not in dataset_points:
        print(f"{dataset_name} not found. Skipping single dataset plot.")
        return

    x = dataset_points[dataset_name]["x"]
    y = dataset_points[dataset_name]["y"]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(x, y, s=10, alpha=0.55, color=color, edgecolors="none")

    ylo, yhi = robust_limits(y)
    ax.set_ylim(ylo, yhi)

    ax.set_title(f"Walking only | {dataset_name}", fontsize=18, pad=10)
    ax.set_xlabel("Sequential sampled points", fontsize=13)
    ax.set_ylabel(dataset_points[dataset_name]["signal_name"], fontsize=13)
    style_axis(ax)

    text_str = (
        f"Files used: {dataset_points[dataset_name]['used_files']}\n"
        f"Points shown: {dataset_points[dataset_name]['n_points']}\n"
        f"Signal: {dataset_points[dataset_name]['signal_name']}"
    )
    ax.text(
        0.99, 0.98, text_str,
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85, edgecolor="0.7")
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_all_datasets_overlay(dataset_points, colors, save_path):
    fig, ax = plt.subplots(figsize=(8, 8))

    total_points = sum(v["n_points"] for v in dataset_points.values())
    all_y = []

    for ds in sorted(dataset_points.keys()):
        x = dataset_points[ds]["x"].copy()
        y = dataset_points[ds]["y"].copy()

        if total_points > MAX_TOTAL_POINTS_ALL_PLOT:
            allowed = max(1000, int(MAX_TOTAL_POINTS_ALL_PLOT * (len(y) / total_points)))
            if len(y) > allowed:
                idx = stable_sample_indices(len(y), allowed, seed=RANDOM_SEED)
                x = x[idx]
                y = y[idx]
                order = np.argsort(x)
                x = x[order]
                y = y[order]

        all_y.append(y)

        ax.scatter(
            x, y,
            s=9,
            alpha=0.35,
            color=colors[ds],
            label=ds,
            edgecolors="none"
        )

    if len(all_y) > 0:
        y_all = np.concatenate(all_y)
        ylo, yhi = robust_limits(y_all)
        ax.set_ylim(ylo, yhi)

    ax.set_title("Walking only | All datasets combined", fontsize=18, pad=10)
    ax.set_xlabel("Sequential sampled points within each dataset", fontsize=13)
    ax.set_ylabel("Selected common signal", fontsize=13)
    style_axis(ax)
    ax.legend(loc="upper right", frameon=True, fontsize=11, markerscale=1.6)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN
# ============================================================
def process_fast_comparison(input_root, output_dir):
    output_dir = resolve_output_dir(output_dir)
    ensure_dir(output_dir)

    dataset_points, summary_rows = collect_fast_dataset_points(input_root)

    if len(dataset_points) == 0:
        print("No usable datasets found.")
        return

    dataset_names = sorted(dataset_points.keys())
    colors = get_dataset_colors(dataset_names)

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(output_dir, "processing_summary_fast.csv")
    summary_df.to_csv(summary_csv, index=False)

    plot_single_dataset(
        dataset_points=dataset_points,
        dataset_name=TARGET_SINGLE_DATASET,
        color=colors.get(TARGET_SINGLE_DATASET, "tab:red"),
        save_path=os.path.join(output_dir, "01_yareta_walking_scatter.png")
    )

    plot_all_datasets_overlay(
        dataset_points=dataset_points,
        colors=colors,
        save_path=os.path.join(output_dir, "02_all_datasets_overlay_scatter.png")
    )

    print("\nDone.")
    print("Output folder:", output_dir)
    print("Summary CSV:", summary_csv)

# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    process_fast_comparison(INPUT_ROOT, OUTPUT_DIR)


Processing dataset: CAMARGO | total files found: 3147
  Walking candidates: 837
  Used files: 12
  Points kept: 12000
  Signal used: L_SHANK_ACC_X

Processing dataset: HUGADB | total files found: 365
  No walking-like files found from file paths.

Processing dataset: IMUVARIOUSWALKING | total files found: 3
  Walking candidates: 3
  Used files: 3
  Points kept: 7500
  Signal used: L_SHANK_ACC_X

Processing dataset: NEWBEE | total files found: 60
  No walking-like files found from file paths.

Processing dataset: RealWorldHAR | total files found: 83
  Walking candidates: 11
  Used files: 11
  Points kept: 12000
  Signal used: L_SHANK_ACC_X

Processing dataset: YARETA | total files found: 367
  Walking candidates: 185
  Used files: 12
  Points kept: 4359
  Signal used: L_SHANK_ACC_X

Done.
Output folder: /Users/parth/Downloads/dataset_comparison_recent
Summary CSV: /Users/parth/Downloads/dataset_comparison_recent/processing_summary_fast.csv
